# 2026.10.14(수) 20h~

## Seq2Seq 모델 정리 (과제: 인코더 / 디코더 / 컨텍스트 벡터 / 교사 강요)

Seq2Seq(Sequence-to-Sequence)는 **순서가 있는 입력(시퀀스)을 받아 순서가 있는 출력(시퀀스)을 만드는 모델**이다. 입력과 출력의 길이가 달라도 되어서 번역, 요약, 챗봇 등에 쓰인다. 크게 **인코더**와 **디코더** 두 개의 RNN(LSTM)으로 이루어진다.

```
입력 시퀀스 ──▶ [ 인코더 LSTM ] ──▶ 컨텍스트 벡터(h, c) ──▶ [ 디코더 LSTM ] ──▶ 출력 시퀀스
  (10개 숫자)     한 칸씩 읽으며 압축     입력 전체의 요약본          요약에서 출발해 한 칸씩 생성   (역순 10개)
```

### 1. 인코더(Encoder)의 기능과 역할
- 입력 시퀀스를 처음부터 끝까지 한 칸씩 읽으면서, 읽은 내용을 **고정된 크기의 상태(h, c)로 압축**한다.
- 중간 출력은 버리고 **마지막 시점의 은닉 상태(h)와 셀 상태(c)만** 디코더에 넘긴다.
- 코드: `LSTM(128, return_state=True)`로 마지막 h, c를 꺼낸다. 입력이 숫자 10개든 문자 15개든, 그 정보가 길이 128짜리 벡터 두 개에 담긴다.

### 2. 디코더(Decoder)의 기능과 역할
- 인코더가 넘겨준 상태를 **시작 상태로 받아서**, 출력 시퀀스를 한 칸씩 만들어 낸다.
- 칸마다 출력을 하나씩 내고, 그 위에 `Dense(…, softmax)`를 붙여 어떤 숫자(문자)인지 확률로 고른다.
- 코드: `LSTM(128, return_sequences=True, return_state=True)`에 `initial_state=encoder_states`를 준다. `return_sequences=True`라서 모든 위치의 출력이 나온다.
- 이 노트북은 `Dense(num_classes=52, softmax)`로 위치마다 52종류 문자 중 하나를 고른다.

### 3. 컨텍스트 벡터(Context Vector)의 의미
- 인코더가 입력 전체를 읽고 만든 **요약본**이다. 이 코드에서는 인코더의 마지막 (h, c)가 그 역할을 한다.
- 디코더는 원래 입력을 다시 볼 수 없다. 출력에 필요한 정보는 이 벡터에 모두 담겨 있어야 한다.
- 입력이 아무리 길어도 **고정된 크기(128)** 의 벡터 하나로 압축된다. 그래서 긴 입력에서는 정보가 손실되는 **병목(bottleneck)** 문제가 생긴다. 이 노트북에서 글자 수를 5 → 15로 늘렸을 때 정확도가 떨어지는 이유 중 하나일 수 있다. 이 병목을 해결하려고 나중에 **Attention** 이 나왔다.

### 4. 교사 강요(Teacher Forcing)의 개념과 역할
- **개념**: 학습할 때 디코더의 다음 입력으로, 모델이 방금 낸 예측 대신 **정답(실제 값)을 넣어 주는 방법**이다.
- **역할**
  - 초반에 예측이 틀려도 그 틀린 값이 다음 단계로 이어지지 않아서, 오류가 쌓이지 않고 **학습이 빠르고 안정적**이다.
  - 매 단계가 "정답을 보고 다음 정답 맞히기"가 되어 학습 신호가 분명해진다.
- **주의점**: 실제 추론에서는 정답이 없으므로 모델이 낸 예측을 다시 입력으로 쓴다(자기회귀). 학습과 추론의 입력이 달라서 생기는 차이를 **노출 편향(exposure bias)** 이라고 한다.
- **이 노트북과의 관계**: 이 코드는 디코더 입력이 `decoder_input_data = np.zeros_like(X)`, 즉 **전부 0**이다. 정답을 넣지도, 예측을 되먹임하지도 않으므로 **교사 강요를 쓰지 않은 단순화된 구조**이다. 디코더는 인코더가 넘긴 (h, c)만으로 역순을 만들어 낸다. 일반적인 Seq2Seq(번역 등)에서는 디코더 입력으로 `<start>` + 정답을 한 칸씩 밀어서 넣는 교사 강요를 쓴다.

---


## 데이터 준비 코드 비교: `3_OnlyLSTM...` vs 이 노트북(`6_..._Seq2Seq_V1.0`)

두 노트북 모두 **영문 대소문자 52종류를 랜덤으로 뽑아 → 번호로 바꾸고 → 역순을 정답으로** 만든다. 큰 틀은 같고, **만드는 방식과 모양(shape)** 이 다르다.

### 1. 한눈에 비교

| 항목 | `3_OnlyLSTM영대소문자_5_10비교_5개` | `6_영대소5_15글자비교_Seq2Seq_V1.0` (이 노트북) |
|---|---|---|
| 글자 수 `cal_len` | 5 | **15** |
| 샘플 수 | 10,000 | **20,000** |
| 문자 목록 `chars`, `num_classes`(52), `char_to_index`, `index_to_char` | 동일 | 동일 |
| 데이터 생성 방식 | 한 번에 **배열로 통째로** (벡터화) | **함수 `generate_data`** 안에서 샘플마다 **반복문** |
| 뽑는 코드 | `np.random.choice(list(chars), size=(10000, cal_len))` | `for _ in range(num_samples): np.random.choice(list(chars), cal_len)` |
| 뽑은 직후 모양 | 2차원 문자 배열 `(10000, 5)` | 샘플 1개씩 1차원 `(15,)` 를 모아서 쌓음 |
| 정답(역순) 만드는 법 | `np.flip(train_data, axis=1)` (배열 뒤집기) | `sequence[::-1]` (슬라이싱으로 뒤집기) |
| 문자 → 번호 변환 | 이중 리스트 컴프리헨션으로 **나중에 한꺼번에** | 샘플을 만들 때 **그 자리에서** 바로 변환 |
| `pad_sequences` | **사용** (`padded_train`, `padded_labels`) | **사용 안 함** |
| 최종 X 모양 | `(10000, 5)` 2차원 | `(20000, 15, 1)` **3차원** |
| 최종 y 모양 | `(10000, 5)` 2차원 | `(20000, 15, 1)` **3차원** |
| 3차원 변환 `reshape(…, 1)` | 없음 | **있음** (LSTM 입력용) |
| 디코더 입력 | 없음 (단일 모델) | `np.zeros_like(X)` (전부 0, 인코더-디코더 구조용) |
| 검증 데이터 | 코드 안에서 20개를 따로 문자열로 생성 | `test_strings` 30개를 문자열로 생성 |

### 2. 같은 점
- 문자 52종류(대문자 26 + 소문자 26), 번호 매핑 사전 두 개(`char_to_index`, `index_to_char`)가 같다.
- `np.random.choice`로 **복원 추출**(같은 글자가 여러 번 나올 수 있음)한다.
- 랜덤 시드를 고정하지 않아서 실행할 때마다 데이터와 결과가 달라진다.
- 입력과 정답이 모두 **번호(정수)** 라서 `sparse_categorical_crossentropy`를 쓴다.

### 3. 다른 점과 그 이유

**(1) 뽑는 방식: 벡터화 vs 반복문**
- 3번은 `size=(10000, 5)`로 한 번에 뽑는다. 코드가 짧고 빠르다.
- 6번은 `generate_data(num_samples)`라는 **함수**로 감싸 샘플마다 반복문으로 만든다. 속도는 느리지만(2만 번 반복) 샘플 하나를 만드는 과정이 눈에 잘 보이고, **나중에 `generate_data(20)`처럼 검증용 데이터를 같은 방식으로 다시 만들 수 있다**(이전 5번 숫자 노트북이 이 패턴).
- 결과(무작위 샘플 + 정답이 역순)는 같다.

**(2) 번호 변환 시점**
- 3번: 문자 배열을 다 만든 뒤 `train_data_indices`, `label_data_indices`로 따로 변환한다 → 중간에 문자 상태 `train_data`가 남아서 출력해 확인하기 좋다.
- 6번: 만드는 도중에 곧바로 번호로 바꿔 리스트에 담는다 → 문자 상태 배열이 따로 남지 않는다.

**(3) `pad_sequences` 유무**
- 3번은 `pad_sequences`를 거치지만 모든 샘플 길이가 같아서 **값은 바뀌지 않는다**(형식만 맞춤).
- 6번은 생략했다. 길이가 항상 `cal_len`으로 같으니 없어도 동작이 같다.

**(4) 모양: 2차원 vs 3차원 ← 가장 큰 차이**
- 3번의 `(N, 5)`는 **Embedding 층**에 넣는 모양이다. Embedding이 번호 하나하나를 100차원 벡터로 바꿔 주므로 번호만 2차원으로 주면 된다.
- 6번의 `(N, 15, 1)`은 **LSTM에 숫자를 그대로** 넣는 모양이다. `(샘플 수, 시간 단계 15, 특성 수 1)` 형태가 필요해서 `reshape(…, 1)`로 마지막에 1을 붙인다. 이 노트북은 Embedding 층이 **없다**.

### 4. 모델에 미치는 영향 (참고)
- 6번은 문자 번호(0~51)를 **크기를 가진 실수처럼** LSTM에 넣는다. 그러면 모델이 `'A'(0)`와 `'z'(51)`을 "51배 차이 나는 값"으로 받아들이게 되는데, 실제로는 문자 사이에 그런 크기 관계가 없다. Embedding은 이 번호를 학습 가능한 벡터로 바꿔 이 문제를 피한다.
- 한 번씩 돌려 본 이 두 노트북의 저장된 출력은 이렇다.

| | 3번 (Embedding + 양방향 LSTM, 5글자) | 6번 (Embedding 없음, 인코더-디코더, 15글자) |
|---|---|---|
| 학습 마지막 정확도(위치별) | 약 100% (val 약 98%) | 약 80% (val 약 75%) |
| 검증 문자열 전체 일치율 | 90% (20개) | 10% (30개) |

- 다만 이 비교는 **길이(5 vs 15), 구조, 샘플 수가 다 달라서** 데이터 표현 방식만의 효과라고 단정할 수는 없다. 글자 수가 길수록 어렵고, 각각 한 번씩만 실행한 결과이며, 랜덤 시드도 고정하지 않았다.
- 같은 조건에서 비교하려면 `cal_len`을 같게 하고(예: 둘 다 5), 시드를 고정하고, 6번에 Embedding을 넣어 보는 식으로 하나씩만 바꿔 보면 된다.

### 5. 이 노트북 코드의 참고 사항
- 3번 셀의 주석 `# 검증용 문자열 리스트 (10글자짜리 30개)`는 `cal_len = 15`와 맞지 않는 옛 주석이다. 실제로는 **15글자짜리 30개**다.
- 위의 "10번 돈다"처럼, 여기서 `15`가 LSTM의 **시간 단계 수**다 (`Input(shape=(cal_len, 1))`). `Dense(num_classes)`의 `52`는 클래스 수로 `cal_len`과 별개다.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

cal_len = 15

# 문자와 인덱스 매핑
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
num_classes = len(chars)

char_to_index = {char: idx for idx, char in enumerate(chars)}
index_to_char = {idx: char for idx, char in enumerate(chars)}

# 데이터 생성 함수
def generate_data(num_samples):
    X = []
    y = []
    for _ in range(num_samples):
        sequence = np.random.choice(list(chars), cal_len)
        X.append([char_to_index[char] for char in sequence])
        y.append([char_to_index[char] for char in sequence[::-1]])
    return np.array(X), np.array(y)

# 데이터 생성
num_samples = 20000
X, y = generate_data(num_samples)

# 데이터 형태 변환 (LSTM을 사용하기 위해 3D 텐서로 변환)
X = X.reshape((X.shape[0], X.shape[1], 1))
y = y.reshape((y.shape[0], y.shape[1], 1))

# 인코더 정의
encoder_inputs = Input(shape=(cal_len, 1))
encoder = LSTM(128, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# 디코더 정의
decoder_inputs = Input(shape=(cal_len, 1))
decoder_lstm = LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_classes, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# 모델 정의
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 입력 데이터와 타겟 데이터를 동일하게 맞춤
decoder_input_data = np.zeros_like(X)

# EarlyStopping 콜백 정의
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 모델 학습
model.fit([X, decoder_input_data], y, epochs=100, validation_split=0.2, callbacks=[early_stopping])



Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.0323 - loss: 3.8495 - val_accuracy: 0.0656 - val_loss: 3.4473
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.0805 - loss: 3.3253 - val_accuracy: 0.1066 - val_loss: 3.0416
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.1199 - loss: 2.9352 - val_accuracy: 0.1441 - val_loss: 2.7172
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1533 - loss: 2.6487 - val_accuracy: 0.1687 - val_loss: 2.4947
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1887 - loss: 2.4264 - val_accuracy: 0.2046 - val_loss: 2.3120
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.2220 - loss: 2.2604 - val_accuracy: 0.2165 - val_loss: 2.2321
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.2490 - loss: 2.1348 - val_accuracy: 0.2525 - val_loss: 2.0750
Epoch 8/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.2766 - loss: 2.0233 - 

In [2]:
# 모델 검증 함수
def predict_reverse(input_sequence):
    input_sequence = np.array([char_to_index[char] for char in input_sequence]).reshape((1, cal_len, 1))
    decoder_input = np.zeros((1, cal_len, 1))
    predicted_sequence = model.predict([input_sequence, decoder_input])
    predicted_indices = np.argmax(predicted_sequence, axis=-1).reshape((cal_len,))
    return ''.join([index_to_char[idx] for idx in predicted_indices])

# 검증용 문자열 리스트 (10글자짜리 30개)
test_strings = ["".join(np.random.choice(list(chars), cal_len)) for _ in range(30)]

# 실제 거꾸로 된 문자열 리스트
expected_outputs = [s[::-1] for s in test_strings]

In [3]:
# 모델 검증 및 정확도 계산
correct_predictions = 0
total_predictions = len(test_strings)

for i, test_string in enumerate(test_strings):
    predicted_output = predict_reverse(test_string)
    is_correct = predicted_output == expected_outputs[i]
    if is_correct:
        correct_predictions += 1
    print(f"입력: {test_string}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {expected_outputs[i]}")
    print(f"정확 여부: {'맞음' if is_correct else '틀림'}\n")

accuracy = correct_predictions / total_predictions
print(f"총 정확도: {accuracy * 100:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
입력: CqRcCIoeJOQlttv
예측된 출력: vstlQOJepICcRpC
실제 출력: vttlQOJeoICcRqC
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
입력: WGffZXfGfrOpXwr
예측된 출력: rwXpOrfGfXZffGW
실제 출력: rwXpOrfGfXZffGW
정확 여부: 맞음

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
입력: beawjCxVHthNkWW
예측된 출력: WVkNhsIVxCiwaeb
실제 출력: WWkNhtHVxCjwaeb
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
입력: EYvHqazmxqiVRgv
예측된 출력: vfRViqxmzbpHuXE
실제 출력: vgRViqxmzaqHvYE
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
입력: udPOtwGfCqUEfyk
예측된 출력: kyfEVqCfGwsOPdu
실제 출력: kyfEUqCfGwtOPdu
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
입력: BxZHBVWwwBhwKaH
예측된 출력: HaKwgBwwXVBIZxB
실제 출력: HaKwhBwwWVBHZxB
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
입력: smgEhIgkzJqfAPB
예측된 출력: BPAfpJzkgIhEfms
실제 출력: BPAfqJzkgIhEgms
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
입력: kdAaohVhiTIXqBC
예측된 출력: CBqXHThhVhoZAdk
실제 출력: CBqXITihVhoaAdk
정확 여부: 틀림

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
입력: XfraYUXAHaVzPlv
예측된 출력: vlPzV

x,y의 데이터 생성 함수를 출력할거고

x는 0,10사이의 수를 샘플값으로 받으며 y는 x의 값에 존속되어 수를 임의 받는다.

2만번의 샘플링으로 x,y의 값을 받게 되며. x, y의 LSTM을 3D 텐서로 변환하기 위해 1,1 부여 받는다.



인코딩과 디코더 모두 LSTM은 128번의 값으로 부여 받는다.

모델 정의는 인코더 디코더의 아웃풋을 받게 되며, 입력 데이터와 타켓 데이터를 동일하게 맞춘다.

이렇게 내가 리뷰하면 되는건가?  - 이렇게 숫자 거꾸로 5개 모델 리뷰를 해봤어요 ㅎㅎ
(동료 수강생 박재은 님 메시지)